In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from collections import Counter


In [ ]:
chunk_size = 7000
block_size = 256
batch_size = 32
n_emb = 256
lr = 1e-3
max_iter = 9000
eval_iters = 200
eval_interval = 1000
num_heads = 8
dk = n_emb//num_heads
eps = 1e-4
adam_eps = 1e-8
num_blocks = 6
TRAINING = True
device = 'mps' if torch.backends.mps.is_available() else 'cpu'
g = torch.Generator().manual_seed(42)


In [ ]:

text = open('corpus.txt', 'r').read()

text = text.replace('’', "'")
cnt = Counter(text)
print(sorted(cnt.items(), key=lambda x: x[1])[:40])

keep = {c for c, n in cnt.items() if n >= 200}
text = ''.join(c for c in text if c in keep)

chars = sorted(list(set(text)))

print(chars)

vocab_size = len(''.join(chars))

stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}

encode = lambda s : [stoi[c] for c in s]
decode = lambda l : ''.join([itos[i] for i in l])

chunks = [text[i:i + chunk_size] for i in range(0, len(text), chunk_size)]

val_chunks = chunks[::10]
train_chunks = [c for i, c in enumerate(chunks) if i % 10 != 0]

train_text = ''.join(train_chunks)
val_text = ''.join(val_chunks)

train_data = torch.tensor(encode(train_text), dtype=torch.long)
val_data = torch.tensor(encode(val_text), dtype=torch.long)

print(len(chunks), len(train_data), len(val_data), vocab_size)


def get_batch(split):
    data = train_data if split == 'train' else val_data
    idx = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in idx])
    y = torch.stack([data[i+1:i+block_size+1] for i in idx])
    x, y = x.to(device), y.to(device)
    return x,y


xb, yb = get_batch('train')

print(xb.shape, yb.shape)



In [ ]:
class Linear :
    
    def __init__(self, fan_in, fan_out):
        self.weight = torch.randn(fan_in, fan_out, generator = g) * (1/ fan_in)**0.5
        self.biais = torch.zeros(fan_out)

    def __call__(self,x):
        out = x @ self.weight + self.biais
        return out
    
    def to(self, device):
        self.weight = self.weight.to(device)
        self.biais = self.biais.to(device)
        return self 

    def parameters(self):
        return [self.weight, self.biais]
    

class Embedding :

    def __init__(self, num_embeddings, dim_emb):
        self.C = torch.randn(num_embeddings, dim_emb, generator=g) * 0.1

    def __call__(self, x):
        out = self.C[x]
        return out

    def to(self, device):
        self.C = self.C.to(device)
        return self

    def parameters(self):
        return [self.C]


class Head : 

    def __init__(self, d):
        self.query = Linear(d, dk)
        self.key = Linear(d, dk)
        self.value = Linear(d, dk)
        self.tril = torch.tril(torch.ones(block_size, block_size))
        self.dropout = Dropout(0.2)

    def __call__(self,x):
        n,p,d = x.shape
        q = self.query(x)
        k = self.key(x)
        v = self.value(x)
        w = q @ k.transpose(-2,-1)
        w = w/(dk**0.5)
        w = w.masked_fill(self.tril[:p , :p]==0, float('-inf'))
        w = F.softmax(w, dim=-1)
        w = self.dropout(w)
        out = w @ v
        return out
    
    def to(self,device):
        self.query = self.query.to(device)
        self.key = self.key.to(device)
        self.value = self.value.to(device)
        self.tril = self.tril.to(device)
        return self

    def parameters(self):
        return self.query.parameters() + self.key.parameters() + self.value.parameters() 
    

class MultiHead :

    def __init__(self):
        self.heads = [Head(n_emb) for _ in range (num_heads)]
        self.weight = Linear(dk*num_heads, n_emb)
        self.dropout = Dropout(0.2)
    
    # dk = n_emb/num_heads
    
    def __call__(self, x):
        h = torch.cat([head(x) for head in self.heads], dim=-1) # n,p, head_size*num_heads
        out = self.weight(h)
        out = self.dropout(out)
        return out

    def to(self, device):
        self.heads = [head.to(device) for head in self.heads]
        self.weight = self.weight.to(device)
        return self

    def parameters(self):
        p = []
        for head in self.heads:
            p += head.parameters()
        return self.weight.parameters() + p


class FFN :

    def __init__(self):
        self.weight1 = Linear(n_emb, 4*n_emb)
        self.weight2 = Linear(4*n_emb, n_emb)
        self.dropout = Dropout(0.2)
    
    def __call__(self, x):
        out = self.weight2(torch.relu(self.weight1(x)))
        out = self.dropout(out)
        return out
    
    def to(self, device):
        self.weight1 = self.weight1.to(device)
        self.weight2 = self.weight2.to(device)
        return self 
    
    def parameters(self):
        return self.weight1.parameters() + self.weight2.parameters()


class LayerNorm :

    def __init__(self, n_emb, eps):
        self.gamma = torch.ones(n_emb)
        self.beta = torch.zeros(n_emb)
        
    def __call__(self, x):
        xmean = x.mean(dim=-1, keepdim=True)
        xvar = x.var(dim=-1, keepdim=True)
        x_norm = (x - xmean)/(xvar + eps)**0.5
        out = x_norm*self.gamma + self.beta
        return out
    
    def to(self, device):
        self.gamma = self.gamma.to(device)
        self.beta = self.beta.to(device)
        return self 

    def parameters(self):
        return [self.gamma, self.beta]


class Block :

    def __init__(self, n_emb):
        self.heads = MultiHead()
        self.ffn = FFN()
        self.layernorm1 = LayerNorm(n_emb, eps)
        self.layernorm2 = LayerNorm(n_emb, eps)
    
    def __call__(self, x):
        x = x + self.heads(self.layernorm1(x))
        x = x + self.ffn(self.layernorm2(x))
        return x
    
    def to(self, device):
        self.heads = self.heads.to(device)
        self.ffn = self.ffn.to(device)
        self.layernorm1 = self.layernorm1.to(device)
        self.layernorm2 = self.layernorm2.to(device)
        return self
    
    def parameters(self):
        return self.heads.parameters() + self.ffn.parameters() + self.layernorm1.parameters() + self.layernorm2.parameters()


class Dropout : 

    def __init__(self, p):
        self.p=p
    def __call__(self,x):
        if not TRAINING :
            return x
        else : 
            mask = (torch.rand(x.shape, device=device) > self.p).float()
            x = x * mask
            x = x/(1-self.p)
            return x
    

class BigramLangageModel :

    def __init__(self, vocab_size, n_emb):
        self.token_embedding = Embedding(vocab_size, n_emb)
        self.position_embedding = Embedding(block_size, n_emb)
        self.blocks = [Block(n_emb) for _ in range(num_blocks)]
        self.layernorm = LayerNorm(n_emb, eps)
        self.linear_head = Linear(n_emb, vocab_size)
        
    def forward(self, idx, targets=None):
        n,p = idx.shape
        x = self.token_embedding(idx) + self.position_embedding(torch.arange(p, device=device))
        for block in self.blocks:
            x = block(x)
        layer_norm_finale = self.layernorm(x)
        logits = self.linear_head(layer_norm_finale) #n,p, vocab_size
        if targets is None :
            loss = None
        else:
            n,p,d = logits.shape 
            logits = logits.view(n*p,d)
            targets = targets.view(n*p)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def to(self, device):
        self.token_embedding = self.token_embedding.to(device) 
        self.position_embedding = self.position_embedding.to(device)
        self.blocks = [block.to(device) for block in self.blocks]
        self.linear_head = self.linear_head.to(device)
        self.layernorm = self.layernorm.to(device)

        return self
    
    def parameters(self):
        p_block = []
        for block in self.blocks:
            p_block += block.parameters()
        return self.token_embedding.parameters() + self.position_embedding.parameters() + self.layernorm.parameters() + self.linear_head.parameters() + p_block

    def eval(self):
        global TRAINING
        TRAINING = False
        return self 
    
    def train(self):
        global TRAINING
        TRAINING = True
        return self


    def generate(self, idx, num_tokens):
        for _ in range (num_tokens):
            logits, loss = self.forward(idx[:, -block_size:])
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_new = torch.multinomial(probs, num_samples = 1)
            idx = torch.cat([idx, idx_new], dim=-1)
        return idx

model = BigramLangageModel(vocab_size, n_emb)
model.to(device)  

In [ ]:
class Adam :
    def __init__(self, beta1, beta2, params, eps, lr):
        self.m = [torch.zeros_like(p) for p in params]
        self.m_hat = [torch.zeros_like(p) for p in params]
        self.v = [torch.zeros_like(p) for p in params]
        self.v_hat = [torch.zeros_like(p) for p in params]
        self.p = list(params)
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = adam_eps
        self.lr = lr
        self.t = 0

    def zero_grad(self):
        for p in self.p: 
            p.grad = None

    def step(self):
        with torch.no_grad():
            self.t += 1
            for k, p in enumerate(self.p):
                
                self.m[k] = self.beta1 * self.m[k] + (1-self.beta1)*(p.grad)
                self.m_hat[k] = (self.m[k])/(1 - self.beta1**(self.t))
                self.v[k] = self.beta2 * self.v[k] + (1-self.beta2)*(p.grad**2)
                self.v_hat[k] = (self.v[k])/(1 - self.beta2**(self.t))
                self.p[k] -= self.lr * self.m_hat[k]/((self.v_hat[k])**0.5 + self.eps)

beta1 = 0.9
beta2 = 0.999

optimizer  = Adam(beta1, beta2, model.parameters(), adam_eps, lr)


for p in model.parameters():
    p.requires_grad = True

@ torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            x, y = get_batch(split)
            logits, loss = model.forward(x, y)
            losses[k] = loss.item()
        out[split] = losses.mean()    
    model.train()
    return out


for iter in range (max_iter):

    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter} : train : {losses['train']} ; val : {losses['val']} ")

    xb, yb = get_batch('train')
    logits, loss = model.forward(xb, yb)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    




In [ ]:
def ecrire(prompt, n=800):
    
    prompt = prompt.replace('’', "'")
    prompt = ''.join(c for c in prompt if c in stoi)
    
    context = torch.tensor([encode(prompt)], dtype=torch.long, device=device)

    model.eval()
    with torch.no_grad():
        out = model.generate(context, n)
    model.train()  

    return decode(out[0].tolist())


print(ecrire("Je"))